### Question 1: Which markets are most efficient, and why?

The definition of market efficiency is how well the market bakes in new information into its prices - to what extent do the prices in the market reflect all available information, and how quickly does the market's prices adjust to new information?  A corollary of this is that it is very difficult for a trader to generate profits because the prices are already "fair value".  

In the case of assets such as stocks, efficiency means that an investor has a hard time beating the overall market benchmark's rate of appreciation (such as S&P500 index's long-term performance of about 7-9% annually)

In the case of daytrading (not investors, those who bet on short-term swings), market efficiency means that they cannot make consistent positive profits because their expect profit E(x) hovers around 0%, especially after transaction costs like broker fees and paying bid/ask spreads

Generally, efficient markets are also highly liquid - they exhibit high trading volumes and narrow bid/ask spreads. This is because when a market has many participants trading in it, the higher chance that a subset of them with crucial information bakes that information into the market prices through their buying/selling actions, and also, the market responds faster to new information.  In other words, high liquidity is correlated with high efficiency, but they are not the same concept.  For instance, here is a degenerate edge case: a market might only have 2 participants, only one buyer and one seller - this is very illiquid.  But, let's say that the true value of a contract is $0.7, and the buyer is bidding $0.60, and the seller is offering at $0.80, which comes out to a mid value of $0.7.  This market is efficient, nonetheless, despite being illiquid.  But this is a very rare in the real financial markets, and I will still use liquidity as one measurement of efficiency.

For prediction markets: I would evaluate efficiency based on these criteria:

Criteria that we *can* test in this dataset:
- **High liquidity**, which has 3 sub bullet points: (1) The bid/ask spread is narrow, (2) the order book is deep, and (3) The trading volume is high. We can assess these criteria with the data that we are given. Which of these 33 markets has the narrowest bid/ask spread, and/or deepest order book, and/or the highest trading volume?
- **Is there arbitrage (risk-less profit opportunity) in the markets?** (4) More details below on the list of mathematical relationships that must hold, or else, an arbitrage exists

Criteria that we *cannot* test in this dataset:
- **Does the price p of a contract actually resolve to "yes" in the long-term p percent of the time?** Given this small dataset, I cannot assert this.  If given time/resources, I would evaluate this by looking the history of baseball game prediction markets, checking the P(yes), and verify how many of those actually ended with resolving as yes
- **Is there a trading strategy that generates profits consistently?** With this small dataset, I cannot say this for sure. If given time/resources, I would try to build valuation models for the "fair value" of these contracts and verify this. I would also verify some simple models like momentum indicators or mean reversion, to test if there is auto-correlation in the time series of these prices
- **Does new information get baked into the market quickly?** I don't have this information in this dataset.  But I would check this by looking at the history of news releases about baseball, i.e. - a star player on a team has been injured, or the betting market about team A vs. team B just received new information that team A beat a team C today, and see how fast these markets adjust accordingly to these news events


### To do: do we use median, mean, or time-weighted for spread and depth??  (For trading volume, it's a straight up sum, no decision required here)

In [1]:
import numpy as np
import pandas as pd

from utils import (median_relative_trading_cost,   # (1) s / (p*(1-p)), median across the window
                   median_order_book_depth,        # (2) qty summed over both sides, median
                   total_contracts_traded)         # (3) qty summed over every trade

# Self-contained: this notebook re-reads the parquet files, so nothing else
# has to be run first.
df_books  = pd.read_parquet("data/orderbook_data_823753_pregame.parquet")
df_trades = pd.read_parquet("data/trades_823753_pregame.parquet")

# All 33 markets, with the shared game stamp stripped off for readability
MARKETS = sorted(df_books["native_id"].unique())
short   = lambda m: m.replace("-26AUG051940PITMIL", "")

print(f"books  {df_books.shape[0]:,} rows across {df_books['native_id'].nunique()} markets")
print(f"trades {df_trades.shape[0]:,} rows across {df_trades['native_id'].nunique()} markets"
      f"  ({len(MARKETS) - df_trades['native_id'].nunique()} markets never traded)")


books  58,275 rows across 33 markets
trades 2,232 rows across 31 markets  (2 markets never traded)


### (1) Trading costs of crossing bid/ask spread

Most of the bid/ask spreads in these markets are \$0.01, so for sake of argument, we'll go with that as our example. Assume that P(yes) = 0.9 in a market, which means P(no) = 0.1. If we buy yes at p = 0.9, we have a max loss of \$0.90. But selling no at \$0.1 is economically identical: we collect \$0.1 premium upfront, but we have a potential liability of \$1 if the bet goes badly, so max loss is also \$0.9 as well. Therefore:

**Buying yes** (equivalently, selling no) deploys $p$ of capital, so the relative cost is

$$\frac{s}{p} \;=\; \frac{0.01}{0.9} \;=\; 1.11\%$$

**Selling yes** (equivalently, buying no) deploys $1-p$ of capital, so the relative cost is

$$\frac{s}{1-p} \;=\; \frac{0.01}{0.1} \;=\; 10.0\%$$

If we want a **direction-neutral** measure of how expensive the market is to trade in bid/ask terms, we should consider both directions and sum them:

$$\frac{s}{p} \;+\; \frac{s}{1-p} \;=\; \frac{s(1-p) + sp}{p(1-p)} \;=\; \frac{s}{p\,(1-p)}$$

which for this example gives

$$\frac{0.01}{0.9 \times 0.1} \;=\; \frac{0.01}{0.09} \;=\; 11.11\% \;=\; 1.11\% + 10.0\% \quad \checkmark$$

Note that the two individual costs are not symmetric — the cheap side of the contract is the expensive side to trade — but their sum is, since $p(1-p)$ is unchanged when $p$ and $1-p$ swap places. That is what makes it a property of the *market* rather than of whichever direction we happened to pick.


Instruction to Claude for trading costs calculation:
- Use the books data (from orderbook_data_823753_pregame.parquet)
- Create a function for one market. In this function: per order book observation/photo (it doesn't matter if msg_type = update or snapshot, because we already established that both are a photo of the order book), calculate the trading cost using the formula above s/(p*(1-p)). For p, just use the mid, which is the average of best_bid and best_offer. For spread s, use difference between best_bid and best_ask: best_ask - best_bid. After you calculate this for every observation/photo of the order book, take the median of the trading cost
- Apply this function for all 33 markets

In [2]:
# (1) Median relative trading cost, one market at a time
cost = pd.Series(
    {short(m): median_relative_trading_cost(df_books[df_books["native_id"] == m])
     for m in MARKETS},
    name = "trading_cost",
)

# Report as a percentage of capital deployed; lower is better
(100 * cost).sort_values().round(2).to_frame("trading_cost_%")


,trading_cost_%
KXMLBF5TOTAL-4,4.00
KXMLBTEAMTOTAL-MIL4,4.00
KXMLBTOTAL-8,4.05
KXMLBTEAMTOTAL-PIT4,4.05
KXMLBTOTAL-7,4.09
KXMLBRFI,4.15
KXMLBTEAMTOTAL-MIL3,4.31
KXMLBF5TOTAL-5,4.31
KXMLBTOTAL-9,4.31
KXMLBTEAMTOTAL-MIL5,4.37


### (2) How deep is the order book?

This is a straightforward check: what is the sum of the quantities of the highest 5 bids and lowest 5 asks for each of these 33 markets?

Instruction to Claude for order book depth calculation:
- Use the books data (from orderbook_data_823753_pregame.parquet)
- Create a function for one market. In this function: per order book observation/photo (it doesn't matter if msg_type = update or snapshot, because we already established that both are a photo of the order book), calculate the sum of the quantities associated with the 5 bids and 5 asks. If the order book has less than 5 bids and 5 asks, just use whatever is available. An order book with only 2 or 3 levels of bids or asks is inherently less liquid, so we would capture that illiquid in the sum, which will be smaller as a result. After you calculate the sum of quantities of the order book for every observation/photo of the order book, take the median
- Apply this function for all 33 markets

In [5]:
# (2) Median order book depth, one market at a time
depth = pd.Series(
    {short(m): median_order_book_depth(df_books[df_books["native_id"] == m])
     for m in MARKETS},
    name = "depth",
)

# Higher is better
depth.sort_values(ascending = False).round(0).to_frame("median_depth")


,median_depth
KXMLBRFI,3232711.0
KXMLBTOTAL-8,444485.0
KXMLBTOTAL-7,340762.0
KXMLBTOTAL-9,283777.0
KXMLBTOTAL-6,190431.0
KXMLBF5TOTAL-4,132731.0
KXMLBTOTAL-10,114164.0
KXMLBTOTAL-11,107641.0
KXMLBTOTAL-12,102596.0
KXMLBTOTAL-5,97518.0


### (3) What is the trading volume in this market?

This is a straightforward check: what is the sum of the trading volume in this market during the timeframe of this dataset?

Instruction to Claude for trading volume calculation:
- Use the trade data (from trades_823753_pregame.parquet)
- Create a function for one market. Simply add up the quantities of all trades across all times in the trade dataframe, for this market.  Every time that you see a trade for this market, add the quantity to a running sum.  At the end, you will have a total number of contracts traded in this timeframe for this market
- Apply this function for all 33 markets.

In [6]:
# (3) Total quantity traded, one market at a time.
# Sliced from the TRADES frame, and a market with no trades at all yields 0.
volume = pd.Series(
    {short(m): total_contracts_traded(df_trades[df_trades["native_id"] == m])
     for m in MARKETS},
    name = "volume",
)

# Higher is better
volume.sort_values(ascending = False).round(2).to_frame("total_volume")


,total_volume
KXMLBRFI,272584.21
KXMLBTOTAL-8,98745.90
KXMLBF5TOTAL-4,35942.38
KXMLBTOTAL-7,28881.61
KXMLBTEAMTOTAL-MIL4,11829.93
KXMLBTOTAL-3,7041.81
KXMLBF5TOTAL-5,5928.34
KXMLBTOTAL-9,4045.00
KXMLBTOTAL-6,3289.39
KXMLBTOTAL-5,2386.03


### All three metrics side by side

One row per market. `trading_cost_%` is a cost, so lower is better; `median_depth`
and `total_volume` are both liquidity, so higher is better. Sorted by volume.


In [7]:
pd.set_option("display.max_rows", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

q1 = pd.DataFrame({
    "trading_cost_%": 100 * cost,
    "median_depth":   depth,
    "total_volume":   volume,
})

q1.sort_values("total_volume", ascending = False)


,trading_cost_%,median_depth,total_volume
KXMLBRFI,4.15,"3,232,711.38","272,584.21"
KXMLBTOTAL-8,4.05,"444,484.61","98,745.90"
KXMLBF5TOTAL-4,4.00,"132,730.82","35,942.38"
KXMLBTOTAL-7,4.09,"340,761.74","28,881.61"
KXMLBTEAMTOTAL-MIL4,4.00,"28,801.27","11,829.93"
KXMLBTOTAL-3,23.27,"91,512.52","7,041.81"
KXMLBF5TOTAL-5,4.31,"44,138.63","5,928.34"
KXMLBTOTAL-9,4.31,"283,777.11","4,045.00"
KXMLBTOTAL-6,4.43,"190,431.31","3,289.39"
KXMLBTOTAL-5,5.73,"97,517.51","2,386.03"


### (4) Checking for Arbitrage

- **Markets have decreasing probabilities p, as number of runs n goes up**: Getting at least 5 runs in a game cannot be more likely than getting at least 4 runs in a game.  Generally speaking, P(runs >= n) is a survival function and must be non-increasing (and most likely decreasing) in number of runs n.  Proof by contradiction: if S(5) = $0.80 and S(4) = $0.75, then we can buy the strike 4 at $0.75, sell the strike 5 at $0.80, earn $0.05 upfront on day 1 (no matter how many runs happen), and if there are exactly 4 runs, we also earn an extra $1 because the 4-strike pays out and the 5-strike does not.  Thus, as a test of market efficiency, we need to check that all of the markets have decreasing probabilities as number of runs n goes up
- **P(KXMLBRFI) ≤ P(KXMLBF5TOTAL runs = 1)**.  
KXMLBRFI = Will there be at least 1 run in the 1st inning of a baseball game (both teams combined total).  
KXMLBF5TOTAL - # of runs in the first 5 innings of the game (both teams combined total).  
Thus, at strike runs = 1 for KXMLBF5TOTAL, we are measuring the probability of there being at least 1 run in the first 5 innings of the game.  This must be at least as great as the probability of at least 1 run in the first inning of the game. 
- **P(KXMLBF5TOTAL) ≤ P(KXMLBTOTAL) for all strikes n**
KXMLBF5TOTAL - # of runs in the first 5 innings of the game (both teams combined total)
KXMLBTOTAL - # of runs scored in the game (both teams combined total)
Same logic as before
- **KXMLBTOTAL >= max[KXMLBTEAMTOTAL(team A), KXMLBTEAMTOTAL(team B)]** In other words, the probability that sum of both teams' runs add up to at least 5 must be at least as great as team A's chance of getting at least 5 runs, and at least as great as team B's chance of getting at least 5 runs

### Sidenote (considered, not computed): speed of repricing

I classified "does new information get baked in quickly?" as untestable here because
there is no news feed in this dataset. That is true for external news, but it is not
the whole story, and I want to flag what I would do with more time.

We have 58,275 book updates across 33 markets that are mechanically related to each
other -- the TOTAL strikes form one ladder, and F5 nests inside TOTAL. So a large
trade landing on one strike is itself an information event, and I can watch whether
the neighbouring strikes reprice, and how many seconds it takes.

That would be the only measure here that tests **efficiency** rather than
**liquidity**. Spread, depth and volume all describe how easy a market is to trade;
none of them show that prices actually absorb information. Repricing speed does, and
it is much harder to fake -- you can post tight quotes in a market nobody watches, but
you cannot make unrelated strikes move in sympathy unless somebody is genuinely
arbitraging them.

Not computed, for time reasons. Noting it so it is clear the omission is a scoping
decision rather than an oversight.
